
# Representative 20-member subset from a 40-member E3SM–DART ensemble

This notebook:

1. discovers paired EAM and ELM files for EN01–EN40;
2. extracts compact atmospheric and land initial-state features;
3. searches reproducibly for a 20-member subset;
4. minimizes mismatch in ensemble mean, spread, and covariance;
5. compares the optimized subset against members 1–20;
6. writes the selected member IDs and diagnostics.

The same member IDs must be used for the paired atmosphere and land states.


In [ ]:
from pathlib import Path
import importlib
import json
import os
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

HELPER_DIR = None
for candidate in (
    Path.cwd().resolve() / "util",
    Path.cwd().resolve() / "ensemble_ic" / "util",
):
    if (candidate / "dart_subset_selection.py").is_file():
        HELPER_DIR = candidate
        break
if HELPER_DIR is None:
    raise FileNotFoundError(
        "Could not find util/dart_subset_selection.py from the notebook working directory."
    )
if str(HELPER_DIR) not in sys.path:
    sys.path.insert(0, str(HELPER_DIR))

import dart_subset_selection as subset_helpers
subset_helpers = importlib.reload(subset_helpers)

from dart_subset_selection import (
    compute_member_features,
    configure_dask,
    discover_member_files,
    improve_by_swaps,
    inventory,
    make_subset_scorer,
    random_search_subsets,
)

warnings.filterwarnings("ignore", category=RuntimeWarning)

machine = "perlmutter" #"compy"
case_name = "DARTEN40S1_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy"

if case_name == "DARTEN40S1_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy":
    N_TOTAL_EXPECTED = 40
    SUBSET_SIZE = 20
    DATE_TAG = "2012-05-16-00000"
elif case_name == "DARTEN40_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy":
    N_TOTAL_EXPECTED = 40
    SUBSET_SIZE = 20
    DATE_TAG = "2011-12-16-00000"
else:
    raise ValueError(f"Unknown case_name: {case_name}")

if machine == "compy":
    REST_DIR = Path(
        "/compyfs/zhan391/v3_dart_cda_scratch/",
        case_name,
        "archive",
        "rest",
        DATE_TAG,
    )
    OUTPUT_DIR = Path("/compyfs/www/zhan391/e3sm_dart/ensemble_ic/dart_subset_selection") / DATE_TAG
elif machine == "perlmutter":
    REST_DIR = Path(
        "/global/cfs/cdirs/m4849/zhan391/e3sm_dart/",
        case_name,
        "archive",
        "rest",
        DATE_TAG,
    )
    OUTPUT_DIR = Path("/global/cfs/cdirs/e3sm/www/zhan391/e3sm_dart_analysis/ensemble_ic/dart_subset_selection") / DATE_TAG

N_TRIALS = 100_000
RANDOM_SEED = 42
MAX_SAMPLE_POINTS = 50_000
MIN_UNIQUE_FEATURE_VALUES = 10
QUANTILES = (0.10, 0.25, 0.50, 0.75, 0.90)

USE_DASK = True
USE_DASK_DISTRIBUTED = False
DASK_N_WORKERS = min(8, os.cpu_count() or 1)
DASK_THREADS_PER_WORKER = 1
DASK_MEMORY_LIMIT = "4GB"
FEATURE_TASK_RETRIES = 1
RANDOM_SEARCH_BATCH_SIZE = 2_000

EAM_VARIABLE_CANDIDATES = ["T", "U", "V", "Q", "PS", "PHIS"]
ELM_VARIABLE_CANDIDATES = [
    "H2OSOI_LIQ", "H2OSOI_ICE", "T_SOISNO",
    "H2OSFC", "SNOWDP", "FSNO", "T_GRND",
]

WEIGHT_MEAN = 1.0
WEIGHT_STD = 1.0
WEIGHT_COV = 0.0

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

dask_client = configure_dask(
    enabled=USE_DASK,
    use_distributed=USE_DASK_DISTRIBUTED,
    n_workers=DASK_N_WORKERS,
    threads_per_worker=DASK_THREADS_PER_WORKER,
    memory_limit=DASK_MEMORY_LIMIT,
)

print("REST_DIR:", REST_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())
print("Dask enabled:", USE_DASK)


## 1. Discover paired EAM and ELM files

In [ ]:
if not REST_DIR.is_dir():
    raise FileNotFoundError(
        f"{REST_DIR} is unavailable. Run this notebook on Compy or edit REST_DIR."
    )

member_files = discover_member_files(REST_DIR, DATE_TAG)

display(member_files.head())
print("Paired members:", len(member_files))

if len(member_files) != N_TOTAL_EXPECTED:
    raise ValueError(
        f"Expected {N_TOTAL_EXPECTED} paired members, found {len(member_files)}."
    )



## 2. Inspect candidate variables

Missing variables are skipped automatically. Modify the lists in the configuration cell after reviewing this inventory.


In [ ]:
eam_inventory = inventory(member_files.loc[0, "eam_file"], EAM_VARIABLE_CANDIDATES)
elm_inventory = inventory(member_files.loc[0, "elm_file"], ELM_VARIABLE_CANDIDATES)

print("EAM variables found")
display(eam_inventory)
print("ELM variables found")
display(elm_inventory)

if eam_inventory.empty or elm_inventory.empty:
    raise ValueError("No usable EAM or ELM candidate variables were found.")



## 3. Extract member-level features

Each field is summarized by its sampled mean, standard deviation, and five quantiles. A deterministic multidimensional stride keeps I/O manageable and samples the same locations for every member.


In [ ]:
eam_variables = eam_inventory["variable"].tolist()
elm_variables = elm_inventory["variable"].tolist()

features_raw = compute_member_features(
    member_files,
    eam_variables=eam_variables,
    elm_variables=elm_variables,
    max_points=MAX_SAMPLE_POINTS,
    quantiles=QUANTILES,
    use_dask=USE_DASK,
    task_retries=FEATURE_TASK_RETRIES,
    min_unique=MIN_UNIQUE_FEATURE_VALUES,
)

feature_summary = pd.DataFrame({
    "raw_mean": features_raw.mean(),
    "raw_std": features_raw.std(ddof=1),
    "raw_min": features_raw.min(),
    "raw_max": features_raw.max(),
    "n_unique": features_raw.nunique(),
})

print("Feature matrix shape:", features_raw.shape)
print("Minimum unique values retained:", MIN_UNIQUE_FEATURE_VALUES)
display(features_raw.head())
print("Feature uniqueness and spread summary")
display(feature_summary.sort_values(["n_unique", "raw_std"]))

features_raw.to_csv(OUTPUT_DIR / "member_features_raw.csv")
feature_summary.to_csv(OUTPUT_DIR / "member_feature_summary.csv")


## 4. Standardize features and define the objective

In [ ]:
features_z = (
    (features_raw - features_raw.mean(axis=0))
    / features_raw.std(axis=0, ddof=1)
)
features_z.to_csv(OUTPUT_DIR / "member_features_standardized.csv")

Z = features_z.to_numpy(dtype=np.float64)
member_ids = features_z.index.to_numpy(dtype=int)

print("Feature matrix shape:", features_raw.shape)
print(
    "Number of unique member-feature rows:",
    np.unique(np.round(Z, 10), axis=0).shape[0],
)
print("Standardized feature matrix")
display(features_z)

subset_score, full_diagnostics = make_subset_scorer(
    Z,
    weight_mean=WEIGHT_MEAN,
    weight_std=WEIGHT_STD,
    weight_cov=WEIGHT_COV,
)
full_mean = full_diagnostics["full_mean"]
full_std = full_diagnostics["full_std"]

first20_indices = np.arange(SUBSET_SIZE)
first20_score, first20_parts = subset_score(first20_indices)
print("Members 1-20:", json.dumps(first20_parts, indent=2))



## 5. Search candidate subsets

There are about \(1.38\times10^{11}\) possible 20-member subsets, so exhaustive search is infeasible. This uses a fixed-seed random search followed by one-member-swap local improvement.


In [ ]:
trial_scores, best_indices, best_score, best_parts = random_search_subsets(
    n_trials=N_TRIALS,
    n_members=len(member_ids),
    subset_size=SUBSET_SIZE,
    subset_score=subset_score,
    random_seed=RANDOM_SEED,
    batch_size=RANDOM_SEARCH_BATCH_SIZE,
    use_dask=USE_DASK,
)

score_diagnostics = pd.DataFrame({
    "quantile": [0, 0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99, 1],
    "score": np.quantile(
        trial_scores,
        [0, 0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99, 1],
    ),
})

print("Best random-search members:", member_ids[best_indices].tolist())
print(json.dumps(best_parts, indent=2))
print("Number of unique random scores:", len(np.unique(np.round(trial_scores, 12))))
print("Score quantiles")
display(score_diagnostics)
score_diagnostics.to_csv(OUTPUT_DIR / "random_search_score_quantiles.csv", index=False)


In [ ]:
optimal_indices, optimal_score, optimal_parts, swap_iterations = improve_by_swaps(
    best_indices,
    n_members=len(member_ids),
    subset_score=subset_score,
)
optimal_members = member_ids[optimal_indices]

print("Optimized members:", optimal_members.tolist())
print("Swap iterations:", swap_iterations)
print(json.dumps(optimal_parts, indent=2))
print("Optimized / members-1-20 score:", optimal_score / first20_score)


## 6. Compare optimized subset, members 1–20, and random subsets

In [ ]:
def save_figure(fig, stem):
    png_path = OUTPUT_DIR / f"{stem}.png"
    pdf_path = OUTPUT_DIR / f"{stem}.pdf"
    fig.savefig(png_path, dpi=150, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    print("Saved figure:", png_path)
    print("Saved figure:", pdf_path)

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(trial_scores, bins=60)
ax.axvline(first20_score, linewidth=2, label="Members 1-20")
ax.axvline(optimal_score, linewidth=2, label="Optimized")
ax.set_xlabel("Objective score")
ax.set_ylabel("Number of candidate subsets")
ax.set_title("Random candidate-subset scores")
ax.legend()
fig.tight_layout()
save_figure(fig, "random_candidate_subset_scores")
plt.show()

def diagnostics(indices, label):
    X = Z[indices]
    return pd.DataFrame({
        "feature": features_z.columns,
        "feature_index": np.arange(Z.shape[1]),
        "subset_mean": X.mean(axis=0),
        "subset_std": X.std(axis=0, ddof=1),
        "label": label,
    })

diag = pd.concat([
    diagnostics(optimal_indices, "Optimized"),
    diagnostics(first20_indices, "Members 1-20"),
], ignore_index=True)

diag.to_csv(OUTPUT_DIR / "subset_feature_diagnostics.csv", index=False)

fig, ax = plt.subplots(figsize=(12, 5))
for label, group in diag.groupby("label"):
    ax.scatter(group["feature_index"], group["subset_mean"], alpha=0.7, label=label)
ax.axhline(0.0, linestyle="--", color="k", linewidth=1)
ax.set_xlabel("Feature index")
ax.set_ylabel("Subset mean in full-ensemble standard deviations")
ax.set_title("Subset mean errors")
ax.legend()
fig.tight_layout()
save_figure(fig, "subset_mean_errors_by_feature")
plt.show()

fig, ax = plt.subplots(figsize=(12, 5))
for label, group in diag.groupby("label"):
    ax.scatter(group["feature_index"], group["subset_std"], alpha=0.7, label=label)
ax.axhline(1.0, linestyle="--", color="k", linewidth=1)
ax.set_xlabel("Feature index")
ax.set_ylabel("Subset spread / full-ensemble spread")
ax.set_title("Subset spread representation")
ax.legend()
fig.tight_layout()
save_figure(fig, "subset_spread_ratio_by_feature")
plt.show()


## 7. Inspect largest feature mismatches and save results

In [ ]:
opt_diag = diagnostics(optimal_indices, "Optimized")
opt_diag["abs_mean_error"] = np.abs(opt_diag["subset_mean"])
opt_diag["abs_spread_error"] = np.abs(opt_diag["subset_std"] - 1.0)
opt_diag["combined_error"] = (
    opt_diag["abs_mean_error"] + opt_diag["abs_spread_error"]
)

display(opt_diag.sort_values("combined_error", ascending=False).head(25))

selection = pd.DataFrame({
    "member": optimal_members,
    "member_label": [f"EN{m:02d}" for m in optimal_members],
})
selection.to_csv(OUTPUT_DIR / "selected_20_members.csv", index=False)
opt_diag.to_csv(OUTPUT_DIR / "optimized_subset_feature_errors.csv", index=False)

metadata = {
    "rest_dir": str(REST_DIR),
    "date_tag": DATE_TAG,
    "n_total": int(len(member_ids)),
    "subset_size": SUBSET_SIZE,
    "n_trials": N_TRIALS,
    "random_seed": RANDOM_SEED,
    "max_sample_points": MAX_SAMPLE_POINTS,
    "min_unique_feature_values": MIN_UNIQUE_FEATURE_VALUES,
    "eam_variables": eam_variables,
    "elm_variables": elm_variables,
    "weights": {
        "mean": WEIGHT_MEAN,
        "std": WEIGHT_STD,
        "covariance": WEIGHT_COV,
    },
    "dask": {
        "enabled": USE_DASK,
        "distributed": USE_DASK_DISTRIBUTED,
        "n_workers": DASK_N_WORKERS,
        "threads_per_worker": DASK_THREADS_PER_WORKER,
        "memory_limit": DASK_MEMORY_LIMIT,
        "feature_task_retries": FEATURE_TASK_RETRIES,
        "random_search_batch_size": RANDOM_SEARCH_BATCH_SIZE,
    },
    "selected_members": optimal_members.tolist(),
    "optimal_score": optimal_parts,
    "members_1_to_20_score": first20_parts,
    "swap_iterations": swap_iterations,
}
with open(OUTPUT_DIR / "selection_metadata.json", "w") as stream:
    json.dump(metadata, stream, indent=2)

print(selection.to_string(index=False))
print("\nSaved to:", OUTPUT_DIR.resolve())



## 8. Cross-date validation before launching the campaign

A subset selected only from `2011-12-21` may be unusually representative of that date. Before finalizing it:

1. run the extraction for at least one additional winter date;
2. run it for at least one late-spring date;
3. test the same selected IDs at every date;
4. inspect mean, spread, and covariance errors.

For the strongest common subset, extract the same features at several dates, standardize them separately within each date, concatenate the standardized date-specific features, and repeat the optimization. That produces one subset representing both winter and late spring.
